# Layer 1; 32; 3x3; AvgPool

In [1]:
import os
print(os.environ.get("LD_LIBRARY_PATH"))

/mnt/c/Users/ASUS/Documents/GitHub/CNN-RNN-LSTM-Scratch/tf-gpu/lib/python3.10/site-packages/nvidia/cublas/lib:/mnt/c/Users/ASUS/Documents/GitHub/CNN-RNN-LSTM-Scratch/tf-gpu/lib/python3.10/site-packages/nvidia/cuda_runtime/lib:/mnt/c/Users/ASUS/Documents/GitHub/CNN-RNN-LSTM-Scratch/tf-gpu/lib/python3.10/site-packages/nvidia/cudnn/lib:/mnt/c/Users/ASUS/Documents/GitHub/CNN-RNN-LSTM-Scratch/tf-gpu/lib/python3.10/site-packages/nvidia/cufft/lib:/mnt/c/Users/ASUS/Documents/GitHub/CNN-RNN-LSTM-Scratch/tf-gpu/lib/python3.10/site-packages/nvidia/cusolver/lib:/mnt/c/Users/ASUS/Documents/GitHub/CNN-RNN-LSTM-Scratch/tf-gpu/lib/python3.10/site-packages/nvidia/cusparse/lib:/mnt/c/Users/ASUS/Documents/GitHub/CNN-RNN-LSTM-Scratch/tf-gpu/lib/python3.10/site-packages/nvidia/nccl/lib:


## Import Libraries

In [2]:
import os

import pandas as pd
import numpy as np
import seaborn as sns

import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))
tf.test.is_built_with_cuda()

I0000 00:00:1778495576.916636    2817 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TF version: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


True

## Config

In [3]:
TRAIN_DIR  = '../../data/intel/seg_train'
VAL_DIR    = '../../data/intel/seg_test'
TEST_DIR   = '../../data/intel/seg_pred'
MODEL_DIR  = '../../data/models/cnn'
os.makedirs(MODEL_DIR, exist_ok=True)

IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
EPOCHS      = 3
NUM_CLASSES = 6
LR          = 1e-3
VAL_SPLIT   = 0.2

## Import Dataset

In [4]:
# datagen = ImageDataGenerator(rescale=1.0/255.0)

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=VAL_SPLIT
)
test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='sparse', subset = 'training', shuffle=True, seed=SEED
)
val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='sparse', subset ='validation', shuffle=False, seed = SEED
)
test_gen =test_datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='sparse', shuffle=False
)

print("Kelas      :", train_gen.class_indices)
print(f"Train      : {train_gen.samples} gambar")
print(f"Validation : {val_gen.samples} gambar")
print(f"Test       : {test_gen.samples} gambar")

Found 11230 images belonging to 6 classes.
Found 2804 images belonging to 6 classes.
Found 3000 images belonging to 6 classes.
Kelas      : {'buildings': 0, 'forest': 1, 'glacier': 2, 'mountain': 3, 'sea': 4, 'street': 5}
Train      : 11230 gambar
Validation : 2804 gambar
Test       : 3000 gambar


## CNN Model Baseline Architecture

In [6]:
MODEL_NAME = 'Layer-1-32-3x3-avgpool'
model = keras.Sequential([
    keras.layers.Input(shape=((224, 224, 3))),
    keras.layers.Conv2D(32, (3, 3), activation='relu', padding='valid', name='conv2s_1'),
    keras.layers.AveragePooling2D((2, 2), name='avgpool_1'),
    keras.layers.Flatten(name='flatten'),
    keras.layers.Dense(128, activation='relu', name='dense_1'),
    keras.layers.Dense(6, activation='softmax', name='output'),
], name=MODEL_NAME)
model.summary()

I0000 00:00:1778496287.977944    2817 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "Layer-1-32-3x3-avgpool"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2s_1 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ avgpool_1 (AveragePooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 394272)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │    50,466,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,468,614 (192.52 MB)

 Trainable params: 50,468,614 (192.52 MB)

 Non-trainable params: 0 (0.00 B)

## Compilation and Training

In [7]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

callback = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience = 5, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath='../../data/models/cnn/baseline_conv2d.h5',
        monitor='val_loss', save_best_only=True, verbose=1
    ),
]

history = model.fit(
    train_gen,
    epochs = EPOCHS,
    validation_data = val_gen,
    callbacks = callback,
    verbose = 1
)

Epoch 1/3


I0000 00:00:1778496303.084020    2817 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1778496306.172912    5212 service.cc:153] XLA service 0x7cc174031ab0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778496306.172986    5212 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9 (Driver: 12.5.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1778496306.384259    5212 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778496306.608626    5212 cuda_dnn.cc:461] Loaded cuDNN version 92200
I0000 00:00:1778496305.260434    5212 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1234__.17
I0000 00:00:1778496324.653197    5212 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the proce

203/351 ━━━━━━━━━━━━━━━━━━━━ 1:19 534ms/step - accuracy: 0.4078 - loss: 7.5033

I0000 00:00:1778496433.855242    5211 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1234__.17
I0000 00:00:1778496437.555278    6343 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_14', 20 bytes spill stores, 20 bytes spill loads



351/351 ━━━━━━━━━━━━━━━━━━━━ 0s 554ms/step - accuracy: 0.4633 - loss: 5.2975
Epoch 1: val_loss improved from None to 0.96818, saving model to ../../data/models/cnn/baseline_conv2d.h5


351/351 ━━━━━━━━━━━━━━━━━━━━ 308s 818ms/step - accuracy: 0.5557 - loss: 1.9822 - val_accuracy: 0.6220 - val_loss: 0.9682
Epoch 2/3
351/351 ━━━━━━━━━━━━━━━━━━━━ 0s 569ms/step - accuracy: 0.6900 - loss: 0.8295
Epoch 2: val_loss improved from 0.96818 to 0.80976, saving model to ../../data/models/cnn/baseline_conv2d.h5


351/351 ━━━━━━━━━━━━━━━━━━━━ 243s 687ms/step - accuracy: 0.7014 - loss: 0.8040 - val_accuracy: 0.7133 - val_loss: 0.8098
Epoch 3/3
 71/351 ━━━━━━━━━━━━━━━━━━━━ 1:43 370ms/step - accuracy: 0.7997 - loss: 0.5796

KeyboardInterrupt: 

## Evaluation

In [ ]:
y_pred_proba = model.predict(test_gen)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = test_gen.classes
macro_f1 = f1_score(y_true, y_pred, average='macro')
print(f"Macro F1 Score: {macro_f1:.4f}")

94/94 ━━━━━━━━━━━━━━━━━━━━ 38s 389ms/step
Macro F1 Score: 0.7361


## Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'],         label='Train Loss')
ax1.plot(history.history['val_loss'],     label='Val Loss')
ax1.set_title(f'{MODEL_NAME} — Loss'); ax1.legend()
ax2.plot(history.history['accuracy'],     label='Train Acc')
ax2.plot(history.history['val_accuracy'], label='Val Acc')
ax2.set_title(f'{MODEL_NAME} — Accuracy'); ax2.legend()
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, f'{MODEL_NAME}_curve.png'), dpi=100)
plt.show()